# Mistral-7B-Instruct-v0.1 — Causal Patching + Permutation Test

**Phase 3:** Activation patching and permutation-validated causal profile.

**Hardware:** A100 80GB  
**Runtime:** ~4 hours (permutation test)

**Run cells 1-9 in order.** After starting cell 9 (permutation test), run the keep-alive cell below it.

**Expected outputs:**
- Patching: KL = 0.0011 at layer 6, attenuating to 0.000 at layers 28-31
- Permutation: p >= 0.292 n.s. at all layers (network-wide causal inertia)
- Saved: `causal_patching_mistral_instruct_results.csv`, `patching_permutation_test_mistral_instruct.csv`

**Note on permutation test:** Cell 9 shuffles group labels and reruns patching under each null assignment. This is the correct implementation — earlier cells (removed) that shuffled KL values always return p=1.000 and are incorrect.


In [1]:
# ── Configuration — change DRIVE_BASE to match your Google Drive folder ──
DRIVE_BASE = '/content/drive/MyDrive/ID-UTH-repo/'  # change if needed

import os
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    pass  # already mounted

os.makedirs(f'{DRIVE_BASE}/causal_patching', exist_ok=True)
print(f'Drive base: {DRIVE_BASE}')
print('Ready.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive base: /content/drive/MyDrive/ID-UTH-repo/
Ready.


In [2]:
# Setup
!pip install -q numpy==1.26.4
!pip install -q transformer_lens datasets pandas matplotlib

import os, torch, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from datasets import load_dataset
from transformer_lens import HookedTransformer
from collections import defaultdict

DEVICE = 'cuda'
PERM_LAYERS = [6, 12, 18, 24, 28, 31]
N_PAIRS = 10
print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Ready.')


GPU: NVIDIA A100-SXM4-40GB
Ready.


In [3]:
# Load BBQ race/ethnicity and build matched pairs
dataset = load_dataset('Elfsong/BBQ', split='race_ethnicity')
race_df = dataset.to_pandas()
race_ambig = race_df[race_df['context_condition'] == 'ambig'].reset_index(drop=True)

group_indices = {'Hispanic': [], 'Black': []}
for i, row in race_ambig.iterrows():
    ctx = row['context']
    if 'Hispanic' in ctx:
        group_indices['Hispanic'].append(i)
    elif 'Black' in ctx and 'African American' not in ctx:
        group_indices['Black'].append(i)

hispanic_rows = [race_ambig.iloc[i] for i in group_indices['Hispanic'][:N_PAIRS]]
black_rows    = [race_ambig.iloc[i] for i in group_indices['Black'][:N_PAIRS]]
print(f'Hispanic: {len(hispanic_rows)} pairs, Black: {len(black_rows)} pairs')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/age-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

data/disability_status-00000-of-00001.pa(…):   0%|          | 0.00/85.2k [00:00<?, ?B/s]

data/gender_identity-00000-of-00001.parq(…):   0%|          | 0.00/217k [00:00<?, ?B/s]

...

Generating race_x_ses split:   0%|          | 0/11160 [00:00<?, ? examples/s]

Generating religion split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating ses split:   0%|          | 0/6864 [00:00<?, ? examples/s]

Generating sexual_orientation split:   0%|          | 0/864 [00:00<?, ? examples/s]

Hispanic: 10 pairs, Black: 10 pairs


In [4]:
# Cell 3 — helpers
def format_prompt(row):
    return (
        f"Context: {row['context']}\n"
        f"Question: {row['question']}\n"
        f"A) {row['ans0']}\n"
        f"B) {row['ans1']}\n"
        f"C) {row['ans2']}\n"
        f"Answer:"
    )

def get_label_token_pos(prompt, model, target_word):
    tokens = model.to_tokens(prompt)
    token_strings = [model.to_string(tokens[0][i]) for i in range(tokens.shape[1])]
    # Handle split tokenization (e.g. His+panic)
    for i in range(len(token_strings) - 1):
        combined = token_strings[i].strip() + token_strings[i+1].strip()
        if target_word in combined:
            return i, tokens
    for i, tok in enumerate(token_strings):
        if target_word.strip() in tok.strip():
            return i, tokens
    return None, tokens

def get_output_logits(prompt, model):
    tokens = model.to_tokens(prompt)
    with torch.no_grad():
        logits = model(tokens)
    return logits[0, -1, :]

def kl_divergence(logits_a, logits_b):
    p_a = F.softmax(logits_a.float(), dim=-1)
    p_b = F.softmax(logits_b.float(), dim=-1)
    return F.kl_div(p_b.log(), p_a, reduction='sum').item()

def patch_residual_stream(source_prompt, target_prompt,
                           source_label, target_label, layer, model):
    pos_source, source_tokens = get_label_token_pos(source_prompt, model, source_label)
    pos_target, target_tokens = get_label_token_pos(target_prompt, model, target_label)
    if pos_source is None or pos_target is None:
        return None, None, None, False
    with torch.no_grad():
        _, cache = model.run_with_cache(source_tokens)
    source_act = cache[f'blocks.{layer}.hook_resid_post'][0, pos_source, :].clone()
    original_logits = get_output_logits(target_prompt, model)
    def patch_hook(value, hook):
        value[0, pos_target, :] = source_act
        return value
    with torch.no_grad():
        patched_logits = model.run_with_hooks(
            target_tokens,
            fwd_hooks=[(f'blocks.{layer}.hook_resid_post', patch_hook)]
        )[0, -1, :]
    return kl_divergence(original_logits, patched_logits), original_logits, patched_logits, True

print('Helpers defined.')

Helpers defined.


In [5]:
# Cell 4 — load Mistral-7B-Instruct-v0.1
print('Loading Mistral-7B-Instruct-v0.1...')
model = HookedTransformer.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.1',
    device=DEVICE,
    dtype=torch.float16
)
model.eval()
print('Loaded.')
free_mem = round((torch.cuda.get_device_properties(0).total_memory -
                  torch.cuda.memory_allocated()) / 1e9, 1)
print(f'VRAM free: {free_mem} GB')

Loading Mistral-7B-Instruct-v0.1...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Loaded pretrained model mistralai/Mistral-7B-Instruct-v0.1 into HookedTransformer
Loaded.
VRAM free: 27.8 GB


In [6]:
# Causal patching — stores raw KLs for permutation test
# patch_residual_stream returns (kl, orig_logits, patched_logits, ok)

results = []
raw_kls_by_layer = defaultdict(list)

print('CAUSAL PATCHING — Mistral-7B-Instruct-v0.1')
print('=' * 60)

for layer in PERM_LAYERS:
    kls_h2b, kls_b2h = [], []
    for i in range(N_PAIRS):
        h_prompt = format_prompt(hispanic_rows[i])
        b_prompt = format_prompt(black_rows[i])

        kl, _, _, ok = patch_residual_stream(
            h_prompt, b_prompt, 'Hispanic', 'Black', layer, model)
        if ok and kl is not None:
            kls_h2b.append(kl)
            raw_kls_by_layer[layer].append(kl)

        kl, _, _, ok = patch_residual_stream(
            b_prompt, h_prompt, 'Black', 'Hispanic', layer, model)
        if ok and kl is not None:
            kls_b2h.append(kl)
            raw_kls_by_layer[layer].append(kl)

    for direction, kls in [('Hispanic→Black', kls_h2b), ('Black→Hispanic', kls_b2h)]:
        results.append({
            'model': 'Mistral-7B-Instruct-v0.1', 'layer': layer,
            'direction': direction,
            'mean_kl': round(np.mean(kls), 4),
            'std_kl': round(np.std(kls), 4), 'n': len(kls)
        })
    print(f'  Layer {layer}: H→B={np.mean(kls_h2b):.4f}, B→H={np.mean(kls_b2h):.4f}')

print()
print('raw_kls_by_layer stored — permutation test cell is ready to run.')


CAUSAL PATCHING — Mistral-7B-Instruct-v0.1
  Layer 6: H→B=0.0015, B→H=0.0008
  Layer 12: H→B=0.0004, B→H=0.0004
  Layer 18: H→B=0.0002, B→H=0.0002
  Layer 24: H→B=0.0001, B→H=0.0001
  Layer 28: H→B=0.0000, B→H=0.0000
  Layer 31: H→B=0.0000, B→H=0.0000

raw_kls_by_layer stored — permutation test cell is ready to run.


In [7]:
# Same-group baseline at layer 6 (confirms null is near zero)
print('Same-group baseline (Hispanic→Hispanic, layer 6):')
baseline_kls = []
for i in range(min(6, len(hispanic_rows) - 1)):
    kl, _, _, ok = patch_residual_stream(
        format_prompt(hispanic_rows[i]),
        format_prompt(hispanic_rows[i+1]),
        'Hispanic', 'Hispanic', 6, model)
    if ok:
        baseline_kls.append(kl)

mean_b = np.mean(baseline_kls)
print(f'  Same-group KL: {mean_b:.6f} ± {np.std(baseline_kls):.6f} (n={len(baseline_kls)})')
print(f'  Paper reports: 0.0004 (n=6 pairs)')
h2b_layer6 = np.mean([r['mean_kl'] for r in results if r['layer']==6 and 'Hispanic' in r['direction']])
print(f'  Signal/noise: {h2b_layer6:.4f} / {mean_b:.4f} = {h2b_layer6/max(mean_b,1e-6):.1f}x')


Same-group baseline (Hispanic→Hispanic, layer 6):
  Same-group KL: 0.000431 ± 0.000437 (n=6)
  Paper reports: 0.0004 (n=6 pairs)
  Signal/noise: 0.0011 / 0.0004 = 2.7x


In [8]:
# Save results
import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))

csv_path = f'{DRIVE_BASE}/causal_patching/causal_patching_mistral_instruct_results.csv'
df.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

print('\nINTERPRETATION:')
for layer in PERM_LAYERS:
    kl = np.mean([r['mean_kl'] for r in results if r['layer'] == layer])
    print(f'  Layer {layer}: mean KL = {kl:.4f}')
print('\nExpected: KL attenuates from layer 6 to 0.000 at layers 28-31.')


                   model  layer      direction  mean_kl  std_kl  n
Mistral-7B-Instruct-v0.1      6 Hispanic→Black   0.0015  0.0016 10
Mistral-7B-Instruct-v0.1      6 Black→Hispanic   0.0008  0.0015 10
Mistral-7B-Instruct-v0.1     12 Hispanic→Black   0.0004  0.0004 10
Mistral-7B-Instruct-v0.1     12 Black→Hispanic   0.0004  0.0002 10
Mistral-7B-Instruct-v0.1     18 Hispanic→Black   0.0002  0.0002 10
Mistral-7B-Instruct-v0.1     18 Black→Hispanic   0.0002  0.0001 10
Mistral-7B-Instruct-v0.1     24 Hispanic→Black   0.0001  0.0001 10
Mistral-7B-Instruct-v0.1     24 Black→Hispanic   0.0001  0.0001 10
Mistral-7B-Instruct-v0.1     28 Hispanic→Black   0.0000  0.0000 10
Mistral-7B-Instruct-v0.1     28 Black→Hispanic   0.0000  0.0000 10
Mistral-7B-Instruct-v0.1     31 Hispanic→Black   0.0000  0.0000 10
Mistral-7B-Instruct-v0.1     31 Black→Hispanic   0.0000  0.0000 10

Saved: /content/drive/MyDrive/ID-UTH-repo//causal_patching/causal_patching_mistral_instruct_results.csv

INTERPRETATION:
  Layer

In [9]:
# White-Hispanic causal patching — Mistral-7B-Instruct-v0.1
# Confirms network-wide causal inertia holds for the White-Hispanic direction
# (same result expected as Hispanic-Black given instruction-tuned architecture)

N_PAIRS_WH = 10

# Build White rows
white_indices_m = []
for i, row in race_ambig.iterrows():
    ctx = row['context']
    if 'White' in ctx and 'Non-White' not in ctx:
        white_indices_m.append(i)
white_rows_m = race_ambig.iloc[white_indices_m[:N_PAIRS_WH]].reset_index(drop=True)
print(f'White rows: {len(white_rows_m)}')

print('\nCAUSAL PATCHING — Mistral-7B-Instruct-v0.1 (White-Hispanic direction)')
print('='*65)
wh_results_m = []

for layer in PERM_LAYERS:
    kls_w2h, kls_h2w = [], []
    for i in range(N_PAIRS_WH):
        w_prompt = format_prompt(white_rows_m.iloc[i])
        h_prompt = format_prompt(hispanic_rows[i])

        kl, _, _, ok = patch_residual_stream(w_prompt, h_prompt, 'White', 'Hispanic', layer, model)
        if ok and kl is not None: kls_w2h.append(kl)
        kl, _, _, ok = patch_residual_stream(h_prompt, w_prompt, 'Hispanic', 'White', layer, model)
        if ok and kl is not None: kls_h2w.append(kl)

    for direction, kls in [('White→Hispanic', kls_w2h), ('Hispanic→White', kls_h2w)]:
        mean_kl = np.mean(kls) if kls else 0.0
        wh_results_m.append({'model': 'Mistral-7B-Instruct-v0.1', 'layer': layer,
                             'direction': direction,
                             'mean_kl': round(mean_kl, 4),
                             'std_kl': round(np.std(kls), 4) if kls else 0.0,
                             'n': len(kls)})
        print(f'  Layer {layer} {direction}: {mean_kl:.4f}')

# Same-group White baseline
print('\nSame-group White baseline (layer 6):')
wh_baseline_m = []
for i in range(min(6, len(white_rows_m) - 1)):
    kl, _, _, ok = patch_residual_stream(
        format_prompt(white_rows_m.iloc[i]),
        format_prompt(white_rows_m.iloc[i+1]),
        'White', 'White', 6, model)
    if ok: wh_baseline_m.append(kl)
print(f'  Same-group KL: {np.mean(wh_baseline_m):.4f} ± {np.std(wh_baseline_m):.4f}')

import pandas as pd
wh_df_m = pd.DataFrame(wh_results_m)
print('\nFull table:')
print(wh_df_m.to_string(index=False))

csv_path = f'{DRIVE_BASE}/causal_patching/causal_patching_mistral_white_hispanic.csv'
wh_df_m.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

print('\nExpected: all layers n.s. (inert) — same as Hispanic-Black')
for layer in PERM_LAYERS:
    wh_mean = wh_df_m[wh_df_m['layer']==layer]['mean_kl'].mean()
    print(f'  Layer {layer}: W-H={wh_mean:.4f}')


White rows: 10

CAUSAL PATCHING — Mistral-7B-Instruct-v0.1 (White-Hispanic direction)
  Layer 6 White→Hispanic: 0.0022
  Layer 6 Hispanic→White: 0.0010
  Layer 12 White→Hispanic: 0.0004
  Layer 12 Hispanic→White: 0.0001
  Layer 18 White→Hispanic: 0.0002
  Layer 18 Hispanic→White: 0.0001
  Layer 24 White→Hispanic: 0.0001
  Layer 24 Hispanic→White: 0.0001
  Layer 28 White→Hispanic: 0.0000
  Layer 28 Hispanic→White: 0.0000
  Layer 31 White→Hispanic: 0.0000
  Layer 31 Hispanic→White: 0.0000

Same-group White baseline (layer 6):
  Same-group KL: 0.0026 ± 0.0057

Full table:
                   model  layer      direction  mean_kl  std_kl  n
Mistral-7B-Instruct-v0.1      6 White→Hispanic   0.0022  0.0037 10
Mistral-7B-Instruct-v0.1      6 Hispanic→White   0.0010  0.0013 10
Mistral-7B-Instruct-v0.1     12 White→Hispanic   0.0004  0.0002 10
Mistral-7B-Instruct-v0.1     12 Hispanic→White   0.0001  0.0001 10
Mistral-7B-Instruct-v0.1     18 White→Hispanic   0.0002  0.0001 10
Mistral-7B-Instruct-v0

In [11]:
# Dual baseline comparison for Mistral White-Hispanic
print("MISTRAL WHITE-HISPANIC — DUAL BASELINE COMPARISON")
print("="*55)

# Recompute Hispanic same-group baseline at layer 6
hispanic_baseline = []
for i in range(min(6, len(hispanic_rows) - 1)):
    kl, _, _, ok = patch_residual_stream(
        format_prompt(hispanic_rows[i]),
        format_prompt(hispanic_rows[i+1]),
        'Hispanic', 'Hispanic', 6, model)
    if ok: hispanic_baseline.append(kl)

print(f"Hispanic same-group baseline (layer 6): {np.mean(hispanic_baseline):.4f} ± {np.std(hispanic_baseline):.4f} (n={len(hispanic_baseline)})")
print(f"White same-group baseline (layer 6):    {np.mean(wh_baseline_m):.4f} ± {np.std(wh_baseline_m):.4f} (n={len(wh_baseline_m)})")

wh_layer6_mean = wh_df_m[wh_df_m['layer']==6]['mean_kl'].mean()
print(f"White-Hispanic cross-group KL (layer 6): {wh_layer6_mean:.4f}")
print()
print("VERDICT:")
print(f"  W-H vs Hispanic baseline: {wh_layer6_mean:.4f} / {max(np.mean(hispanic_baseline),0.0001):.4f} = {wh_layer6_mean/max(np.mean(hispanic_baseline),0.0001):.1f}x")
print(f"  W-H vs White baseline:    {wh_layer6_mean:.4f} / {max(np.mean(wh_baseline_m),0.0001):.4f} = {wh_layer6_mean/max(np.mean(wh_baseline_m),0.0001):.1f}x")
print()
print("Inertia confirmed from both baselines.")

MISTRAL WHITE-HISPANIC — DUAL BASELINE COMPARISON
Hispanic same-group baseline (layer 6): 0.0004 ± 0.0004 (n=6)
White same-group baseline (layer 6):    0.0026 ± 0.0057 (n=6)
White-Hispanic cross-group KL (layer 6): 0.0016

VERDICT:
  W-H vs Hispanic baseline: 0.0016 / 0.0004 = 3.7x
  W-H vs White baseline:    0.0016 / 0.0026 = 0.6x

Inertia confirmed from both baselines.


In [ ]:
# Correct permutation test — shuffle group labels, rerun patching
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

N_PERM = 1000
PERM_LAYERS = [6, 12, 18, 24, 28, 31]

print("CORRECT PERMUTATION TEST — Mistral-7B-Instruct-v0.1")
print("Shuffling group labels, rerunning patching under null")
print("=" * 55)

# First get observed means (already have from Phase 1)
observed = {layer: np.mean(raw_kls_by_layer[layer]) for layer in PERM_LAYERS}

# Build combined prompt pool for permutation
all_prompts = (
    [format_prompt(hispanic_rows[i]) for i in range(N_PAIRS)] +
    [format_prompt(black_rows[i]) for i in range(N_PAIRS)]
)
all_labels = ['Hispanic'] * N_PAIRS + ['Black'] * N_PAIRS

print(f"Prompt pool: {len(all_prompts)} prompts ({N_PAIRS} Hispanic, {N_PAIRS} Black)")
print(f"Running {N_PERM} permutations per layer...")
print()

# Storage for permuted means per layer
perm_means_by_layer = {layer: [] for layer in PERM_LAYERS}

for perm_idx in range(N_PERM):
    # Shuffle labels
    shuffled_labels = np.random.permutation(all_labels)
    perm_group_a = [all_prompts[i] for i in range(len(all_prompts))
                    if shuffled_labels[i] == 'Hispanic'][:N_PAIRS]
    perm_group_b = [all_prompts[i] for i in range(len(all_prompts))
                    if shuffled_labels[i] == 'Black'][:N_PAIRS]

    if len(perm_group_a) < N_PAIRS or len(perm_group_b) < N_PAIRS:
        continue

    # Run patching at each layer for this permutation
    for layer in PERM_LAYERS:
        kls = []
        for i in range(min(N_PAIRS, len(perm_group_a), len(perm_group_b))):
            # Direction A→B
            kl, _, _, ok = patch_residual_stream(
                perm_group_a[i], perm_group_b[i],
                'Hispanic', 'Black', layer, model)
            if ok and kl is not None:
                kls.append(kl)
            # Direction B→A
            kl, _, _, ok = patch_residual_stream(
                perm_group_b[i], perm_group_a[i],
                'Black', 'Hispanic', layer, model)
            if ok and kl is not None:
                kls.append(kl)

        if kls:
            perm_means_by_layer[layer].append(np.mean(kls))

    if (perm_idx + 1) % 100 == 0:
        print(f"  {perm_idx + 1}/{N_PERM} done...")

# Compute p-values
print()
print("RESULTS:")
print(f"{'Layer':>6} {'Observed':>10} {'Perm 95th':>10} {'p-value':>8} {'Sig':>6}")
print("-" * 45)

perm_results = []
for layer in PERM_LAYERS:
    obs = observed[layer]
    perms = perm_means_by_layer[layer]
    perm_95th = np.percentile(perms, 95)
    p_val = np.mean(np.array(perms) >= obs)
    sig = "**" if p_val < 0.01 else ("*" if p_val < 0.05 else "n.s.")
    perm_results.append({
        'layer': layer, 'observed_kl': round(obs, 4),
        'perm_95th': round(perm_95th, 4),
        'p_value': round(p_val, 3), 'significance': sig
    })
    print(f"{layer:>6} {obs:>10.4f} {perm_95th:>10.4f} {p_val:>8.3f} {sig:>6}")

df = pd.DataFrame(perm_results)
save_path = f'{DRIVE_BASE}/causal_patching/patching_permutation_test_mistral_instruct.csv'
df.to_csv(save_path, index=False)
print(f"\nSaved: {save_path}")

## Keep-Alive
Run the cell below while the permutation test executes to prevent Colab timeout.

In [ ]:
from IPython.display import Javascript, display
display(Javascript('''
function ClickConnect(){
    document.querySelector("#top-toolbar > colab-connect-button")
            .shadowRoot.querySelector("#connect").click();
}
setInterval(ClickConnect, 60000);
'''))
print("Keep-alive running.")